# 02. Data-quality log

**Purpose:** Profile every table (row counts, duplicates, nulls), check the order timeline and join logic, and count how many orders are usable for delivery analysis.

**Inputs:** the 9 MySQL tables created by notebook 01.

**Outputs:** `docs/dq_table_summary.csv`, `docs/dq_null_counts.csv`, `docs/dq_order_status.csv`, `docs/dq_logic_checks.csv` (counts only, no raw data).

**Run order:** after notebook 01.

## 1. Setup

In [1]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent
load_dotenv(ROOT / ".env")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)
DOCS = ROOT / "docs"
DOCS.mkdir(exist_ok=True)

def run(sql):
    """Run a SQL query and return the result as a pandas table."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

tables = ["orders", "order_items", "order_payments", "order_reviews", "customers",
          "sellers", "products", "geolocation", "category_translation"]

## 2. Row counts, duplicate rows and nulls per column

Geolocation is large, so this cell takes a minute.

In [2]:
summary, nulls = [], []
for t in tables:
    df = run(f"SELECT * FROM {t}")
    summary.append({"table": t, "rows": len(df), "columns": df.shape[1],
                    "fully_duplicated_rows": int(df.duplicated().sum())})
    for col, n in df.isna().sum().items():
        if n > 0:
            nulls.append({"table": t, "column": col, "null_count": int(n),
                          "null_pct": round(100 * n / len(df), 2)})

summary_df, nulls_df = pd.DataFrame(summary), pd.DataFrame(nulls)
summary_df.to_csv(DOCS / "dq_table_summary.csv", index=False)
nulls_df.to_csv(DOCS / "dq_null_counts.csv", index=False)

print("TABLE SUMMARY")
display(summary_df)
print("COLUMNS WITH NULLS")
display(nulls_df)

TABLE SUMMARY


,table,rows,columns,fully_duplicated_rows
0,orders,99441,8,0
1,order_items,112650,7,0
2,order_payments,103886,5,0
3,order_reviews,99224,7,0
4,customers,99441,5,0
5,sellers,3095,4,0
6,products,32951,9,0
7,geolocation,1000163,5,261831
8,category_translation,71,2,0


COLUMNS WITH NULLS


,table,column,null_count,null_pct
0,orders,order_approved_at,160,0.16
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_delivered_customer_date,2965,2.98
3,order_reviews,review_comment_title,87656,88.34
4,order_reviews,review_comment_message,58247,58.70
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01


## 3. Order status distribution

In [3]:
status = run("""
    SELECT order_status, COUNT(*) AS orders,
           ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM orders), 2) AS pct
    FROM orders GROUP BY order_status ORDER BY orders DESC
""")
status.to_csv(DOCS / "dq_order_status.csv", index=False)
display(status)

,order_status,orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


## 4. Logic and integrity checks

Each check counts rows that break a business rule (impossible timestamps, contradictory status, orphan rows) or that would cause double counting when joined (several items, sellers, payments or reviews per order).

In [4]:
checks = {
    "duplicate order_id in orders":
        "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders",
    "approved before purchase":
        "SELECT COUNT(*) FROM orders WHERE order_approved_at < order_purchase_timestamp",
    "carrier pickup before purchase":
        "SELECT COUNT(*) FROM orders WHERE order_delivered_carrier_date < order_purchase_timestamp",
    "customer delivery before purchase":
        "SELECT COUNT(*) FROM orders WHERE order_delivered_customer_date < order_purchase_timestamp",
    "customer delivery before carrier pickup":
        "SELECT COUNT(*) FROM orders WHERE order_delivered_customer_date < order_delivered_carrier_date",
    "estimated delivery before purchase":
        "SELECT COUNT(*) FROM orders WHERE order_estimated_delivery_date < order_purchase_timestamp",
    "status delivered but no delivery date":
        "SELECT COUNT(*) FROM orders WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL",
    "status not delivered but has delivery date":
        "SELECT COUNT(*) FROM orders WHERE order_status <> 'delivered' AND order_delivered_customer_date IS NOT NULL",
    "order_items with no matching order":
        "SELECT COUNT(*) FROM order_items oi LEFT JOIN orders o ON o.order_id = oi.order_id WHERE o.order_id IS NULL",
    "orders with no items":
        "SELECT COUNT(*) FROM orders o LEFT JOIN order_items oi ON o.order_id = oi.order_id WHERE oi.order_id IS NULL",
    "orders with more than one seller":
        "SELECT COUNT(*) FROM (SELECT order_id FROM order_items GROUP BY order_id HAVING COUNT(DISTINCT seller_id) > 1) x",
    "orders with more than one payment row":
        "SELECT COUNT(*) FROM (SELECT order_id FROM order_payments GROUP BY order_id HAVING COUNT(*) > 1) x",
    "orders with more than one review":
        "SELECT COUNT(*) FROM (SELECT order_id FROM order_reviews GROUP BY order_id HAVING COUNT(*) > 1) x",
    "duplicate review_id values":
        "SELECT COUNT(*) - COUNT(DISTINCT review_id) FROM order_reviews",
    "review score outside 1-5":
        "SELECT COUNT(*) FROM order_reviews WHERE review_score NOT BETWEEN 1 AND 5",
    "items with price <= 0":
        "SELECT COUNT(*) FROM order_items WHERE price <= 0",
    "customer_id rows vs unique people":
        "SELECT COUNT(*) - COUNT(DISTINCT customer_unique_id) FROM customers",
    "products with category not in translation table":
        "SELECT COUNT(*) FROM products p LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name "
        "WHERE p.product_category_name IS NOT NULL AND t.product_category_name IS NULL",
}

results = [{"check": name, "rows_affected": int(run(sql).iloc[0, 0])} for name, sql in checks.items()]
results_df = pd.DataFrame(results)
results_df.to_csv(DOCS / "dq_logic_checks.csv", index=False)
display(results_df)

,check,rows_affected
0,duplicate order_id in orders,0
1,approved before purchase,0
2,carrier pickup before purchase,166
3,customer delivery before purchase,0
4,customer delivery before carrier pickup,23
5,estimated delivery before purchase,0
6,status delivered but no delivery date,8
7,status not delivered but has delivery date,6
8,order_items with no matching order,0
9,orders with no items,775


## 5. Orders usable for delivery analysis

An order is usable if it is `delivered`, has both delivery dates, was picked up by the carrier after purchase, and was delivered after pickup.

In [5]:
usable = run("""
    SELECT COUNT(*) FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_delivered_carrier_date IS NOT NULL
      AND order_delivered_carrier_date >= order_purchase_timestamp
      AND order_delivered_customer_date >= order_delivered_carrier_date
""").iloc[0, 0]
total = run("SELECT COUNT(*) FROM orders").iloc[0, 0]
print(f"Usable delivered orders for delivery metrics: {usable:,} of {total:,} ({100*usable/total:.2f}%)")

Usable delivered orders for delivery metrics: 96,281 of 99,441 (96.82%)


## Findings

- 99,441 orders, of which 96,478 are `delivered` and 2,963 are not.
- **96,281 orders (96.82% of all orders, 99.8% of delivered orders) are usable for delivery metrics.** The 197 delivered orders dropped are exactly 166 (carrier pickup before purchase) + 23 (delivered before pickup) + 8 (status delivered but no delivery date).
- Geolocation has 261,831 fully duplicated rows out of 1,000,163, so coordinates are averaged per zip prefix before use.
- Join fan-out risks: 1,278 orders have more than one seller, 2,961 more than one payment row, 547 more than one review, and 814 `review_id` values repeat. Items, payments and reviews are therefore aggregated to one row per order before joining.
- 775 orders have no items; 3,345 `customer_id` rows belong to returning people, so repeat purchase uses `customer_unique_id`.
- 610 products have no category and 13 have a category missing from the translation table.

The resulting exclusion decisions are in the README.